# 03 — Challenge solution

Debrief only. Same kernel as the student notebook, after `get_fact`, `get_flight`, `tools`, and `run_official_call` exist.

The official loop, pointed at Tokyo → Moscow vs Tokyo → Berlin. Moscow is cheaper (259.3 vs 346.16). Then: Moscow's metro stations are often referred to as 'underground palaces'. Three tool results is the honest chain.


In [22]:
messages = [
    {
        "role": "user",
        "content": "From Tokyo, is it cheaper to fly to Moscow or to Berlin? Give me a fun fact about whichever one is cheaper.",
    }
]
n_lookups = 0
final_text = None

for turn in range(6):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        tools=tools,
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    message = response.choices[0].message
    print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")

    if not message.tool_calls:
        final_text = message.content
        print(final_text)
        break

    messages.append(message)
    for call in message.tool_calls:
        result = run_official_call(call)
        n_lookups = n_lookups + 1
        print(call.function.name, "->", result)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

print("n_lookups:", n_lookups)


--- turn 1 finish_reason: tool_calls ---
get_flight -> 259.3 dollars, 525 minutes
get_flight -> 346.16 dollars, 609 minutes


--- turn 2 finish_reason: tool_calls ---
get_fact -> Moscow's metro stations are often referred to as 'underground palaces'.


--- turn 3 finish_reason: stop ---
From **Tokyo**, it’s cheaper to fly to **Moscow** than to **Berlin**.

- **Tokyo → Moscow:** **$259.30** (525 minutes)  
- **Tokyo → Berlin:** **$346.16** (609 minutes)

**Fun fact (Moscow):** Moscow’s metro stations are often nicknamed **“underground palaces.”**
n_lookups: 3


In [23]:
assert n_lookups >= 1, "the loop should have run at least one tool"
assert final_text and str(final_text).strip(), "final_text should be the last model sentence"
print("looks good")

looks good
